# 🏆 Amazon ML Challenge 2026
## Business Entity Resolution — Full End-to-End Pipeline

**Pipeline Steps:**
1. Install Dependencies
2. Upload Datasets
3. DataLoader
4. DataCleaner
5. CandidateGenerator
6. FeatureEngineer
7. TrainingPairGenerator
8. Train LightGBM
9. Optimize Threshold
10. Save & Download Outputs

---
## Cell 1 — Install Dependencies

In [ ]:
!pip install -q rapidfuzz lightgbm tqdm
import importlib
for pkg in ['pandas','numpy','sklearn','rapidfuzz','lightgbm','tqdm','joblib']:
    try:
        importlib.import_module(pkg)
        print(f'  ✅  {pkg}')
    except ImportError:
        print(f'  ❌  {pkg} MISSING')

---
## Cell 2 — Create Output Directories

In [ ]:
import os
os.makedirs('output', exist_ok=True)
os.makedirs('models', exist_ok=True)
print('✅ output/ and models/ directories ready')

---
## Cell 3 — Upload Dataset Files
Upload these 4 files when prompted:
- `train_source1.tsv`
- `train_source2.tsv`
- `train_source3.tsv`
- `train_ground_truth.tsv`

In [ ]:
from google.colab import files
uploaded = files.upload()
print('\nUploaded files:')
for name in uploaded:
    size_mb = len(uploaded[name]) / 1_000_000
    print(f'  {name}  ({size_mb:.1f} MB)')

---
## Cell 4 — DataLoader: Load & Validate All Datasets
*Note: Limiting to 100,000 rows for fast testing.*

In [ ]:
import pandas as pd
import sys

SOURCE_COLUMNS     = ['entity_id', 'business_name', 'business_address', 'country']
GT_COLUMNS         = ['source1_entity_id', 'matched_entity_ids']

def load_tsv(path, expected_cols=None, label='', nrows=100000):
    """Load a TSV file, validate columns, and print stats."""
    print(f'\n{'='*65}')
    print(f'Loading: {label}  ({path})')
    print(f'{'='*65}')
    df = pd.read_csv(path, sep='\t', dtype=str, encoding='utf-8', nrows=nrows)
    print(f'  Shape   : {df.shape[0]:,} rows  x  {df.shape[1]} columns')
    print(f'  Columns : {df.columns.tolist()}')
    if expected_cols:
        missing = [c for c in expected_cols if c not in df.columns]
        if missing:
            raise ValueError(f'Missing columns: {missing}')
        print(f'  ✅ Column validation passed')
    missing_vals = df.isnull().sum()
    missing_vals = missing_vals[missing_vals > 0]
    if len(missing_vals):
        print(f'  ⚠️  Missing values:')
        for col, cnt in missing_vals.items():
            print(f'       {col}: {cnt:,}  ({cnt/len(df)*100:.2f}%)')
    else:
        print(f'  ✅ No missing values')
    print(f'  Sample:')
    print(df.head(3).to_string(index=False))
    return df

df1 = load_tsv('train_source1.tsv', SOURCE_COLUMNS, 'Source 1')
df2 = load_tsv('train_source2.tsv', SOURCE_COLUMNS, 'Source 2')
df3 = load_tsv('train_source3.tsv', SOURCE_COLUMNS, 'Source 3')
gt  = load_tsv('train_ground_truth.tsv', GT_COLUMNS,  'Ground Truth')

---
## Cell 5 — DataCleaner: Clean Names, Addresses & Countries

In [ ]:
import re, string

BUSINESS_NORMS = {
    r'\bpvt\b':  'private',
    r'\bltd\b':  'limited',
    r'\bcorp\b': 'corporation',
    r'\bco\b':   'company',
    r'\binc\b':  'incorporated',
    r'\bllc\b':  'llc',
}
ADDRESS_NORMS = {
    r'\brd\b':   'road',
    r'\bst\b':   'street',
    r'\bave\b':  'avenue',
    r'\bblvd\b': 'boulevard',
    r'\bdr\b':   'drive',
    r'\bln\b':   'lane',
}

def _apply_norms(text, norms):
    for pattern, replacement in norms.items():
        text = re.sub(pattern, replacement, text)
    return text

def clean_name(v):
    if pd.isna(v) or not isinstance(v, str): return ''
    v = v.lower().translate(str.maketrans('', '', string.punctuation))
    v = _apply_norms(v, BUSINESS_NORMS)
    return re.sub(r'\s+', ' ', v).strip()

def clean_address(v):
    if pd.isna(v) or not isinstance(v, str): return ''
    v = v.lower().translate(str.maketrans('', '', string.punctuation))
    v = _apply_norms(v, ADDRESS_NORMS)
    return re.sub(r'\s+', ' ', v).strip()

def clean_country(v):
    if pd.isna(v) or not isinstance(v, str): return ''
    return re.sub(r'\s+', ' ', v.lower().strip())

def clean_df(df):
    d = df.copy()
    d['business_name']    = d['business_name'].apply(clean_name)
    d['business_address'] = d['business_address'].apply(clean_address)
    d['country']          = d['country'].apply(clean_country)
    return d

print('Cleaning Source1...')
df1c = clean_df(df1)
print('Cleaning Source2...')
df2c = clean_df(df2)
print('Cleaning Source3...')
df3c = clean_df(df3)

# Combine Source2 + Source3 into a single target pool
df_target = pd.concat([df2c, df3c], ignore_index=True)

print(f'\n✅ Cleaning complete')
print(f'  Source1  : {df1c.shape}')
print(f'  Source2  : {df2c.shape}')
print(f'  Source3  : {df3c.shape}')
print(f'  Target (S2+S3): {df_target.shape}')
print(f'\nCleaned sample:')
print(df1c.head(3).to_string(index=False))

---
## Cell 6 — CandidateGenerator (Memory-Efficient)
**Blocking Rules:**
1. Same `country`
2. Same first character of `business_name`
3. At least one common word token in `business_name`

In [ ]:
from collections import defaultdict
from tqdm import tqdm
import csv

def build_s1_index(df, id_col='entity_id'):
    """Build blocking index on Source 1 only: (country_firstchar) -> list of {id, tokens}."""
    index = defaultdict(list)
    for _, row in tqdm(df.iterrows(), total=len(df), desc='  Indexing Source 1'):
        country = str(row.get('country', '')).strip()
        name    = str(row.get('business_name', '')).strip()
        if not country or not name: continue
        key = f"{country}_{name[0]}"
        index[key].append({'id': row[id_col], 'tokens': set(name.split())})
    return index

def probe_and_write(df_target, index, output_csv, target_name, mode='a', write_header=False, chunk_size=100000):
    """Probe S1 index with Target chunks and write directly to CSV."""
    total = 0
    with open(output_csv, mode, newline='', encoding='utf-8') as f:
        writer = csv.writer(f)
        if write_header: writer.writerow(['source1_entity_id', 'target_entity_id'])
        
        for start in tqdm(range(0, len(df_target), chunk_size), desc=f'  Probing {target_name}'):
            chunk = df_target.iloc[start:start+chunk_size]
            pairs = []
            for _, row in chunk.iterrows():
                country = str(row.get('country', '')).strip()
                name    = str(row.get('business_name', '')).strip()
                if not country or not name: continue
                key = f"{country}_{name[0]}"
                if key not in index: continue
                tgt_tokens = set(name.split())
                tgt_id = row['entity_id']
                for m in index[key]:
                    if tgt_tokens & m['tokens']:
                        pairs.append([m['id'], tgt_id])
            if pairs:
                writer.writerows(pairs)
                total += len(pairs)
    return total

out_path = 'output/candidate_pairs.csv'
print('Building index for Source 1 (Memory Efficient)...')
s1_idx = build_s1_index(df1c)

print('Probing Source 2 vs Source 1...')
total_s2 = probe_and_write(df2c, s1_idx, out_path, 'Source 2', mode='w', write_header=True)
print('Probing Source 3 vs Source 1...')
total_s3 = probe_and_write(df3c, s1_idx, out_path, 'Source 3', mode='a', write_header=False)

print('\nLoading compact CSV to deduplicate...')
candidates = pd.read_csv(out_path, dtype=str)
before_dedup = len(candidates)
candidates = candidates.drop_duplicates()
candidates.to_csv(out_path, index=False)

print(f'\n✅ CandidateGenerator complete')
print(f'  Source 2 matches : {total_s2:,}')
print(f'  Source 3 matches : {total_s3:,}')
print(f'  Total (deduped)  : {len(candidates):,} (from {before_dedup:,})')
print(candidates.head(5).to_string(index=False))

---
## Cell 7 — TrainingPairGenerator: Parse Ground Truth & Build Labels

In [ ]:
# Parse ground truth — one-to-many: explode comma-separated matched_entity_ids
gt_valid = gt.dropna(subset=['matched_entity_ids']).copy()
print(f'Ground Truth: {len(gt):,} total rows, {len(gt_valid):,} with matches, {len(gt)-len(gt_valid):,} with no match')

positive_pairs = []
for _, row in tqdm(gt_valid.iterrows(), total=len(gt_valid), desc='Exploding GT'):
    src1_id = row['source1_entity_id']
    for tid in str(row['matched_entity_ids']).split(','):
        tid = tid.strip()
        if tid:
            positive_pairs.append({'source1_entity_id': src1_id, 'target_entity_id': tid, 'label': 1})

positives_df = pd.DataFrame(positive_pairs).drop_duplicates()
print(f'Positive pairs (label=1): {len(positives_df):,}')

if candidates.empty:
    print('⚠️ No candidate pairs generated. Cannot build negative pairs.')
    negatives_df = pd.DataFrame(columns=['source1_entity_id', 'target_entity_id', 'label'])
else:
    # Negatives = candidates NOT in positives (left anti-join)
    merged = candidates.merge(
        positives_df[['source1_entity_id','target_entity_id']],
        on=['source1_entity_id','target_entity_id'],
        how='left', indicator=True
    )
    negatives_df = merged[merged['_merge']=='left_only'][['source1_entity_id','target_entity_id']].copy()
    negatives_df['label'] = 0
print(f'Negative pairs (label=0): {len(negatives_df):,}')

# Balance: downsample negatives to match positives count
n_pos = len(positives_df)
if len(negatives_df) > n_pos and n_pos > 0:
    negatives_df = negatives_df.sample(n=n_pos, random_state=42)
    print(f'Downsampled negatives to: {len(negatives_df):,}')

# Combine & shuffle
training_pairs = pd.concat(
    [positives_df, negatives_df], ignore_index=True
).sample(frac=1, random_state=42).reset_index(drop=True)

print(f'\n✅ Training pairs ready')
print(f'  Total        : {len(training_pairs):,}')
print(f'  Positives    : {training_pairs.label.sum():,}')
print(f'  Negatives    : {(training_pairs.label==0).sum():,}')

---
## Cell 8 — FeatureEngineer: Fit TF-IDF & Compute All Features

In [ ]:
import numpy as np
from rapidfuzz import fuzz
from sklearn.feature_extraction.text import TfidfVectorizer

if training_pairs.empty:
    raise ValueError("No training pairs available. Try adjusting subset size or check data distribution.")

# Fit TF-IDF on full name and address corpus
print('Fitting TF-IDF vectorizers on full corpus...')
name_vec = TfidfVectorizer(analyzer='char_wb', ngram_range=(2, 4))
addr_vec = TfidfVectorizer(analyzer='char_wb', ngram_range=(2, 4))

all_names = pd.concat([
    df1c['business_name'], df_target['business_name']
]).fillna('').astype(str)
all_addrs = pd.concat([
    df1c['business_address'], df_target['business_address']
]).fillna('').astype(str)

name_vec.fit(all_names)
addr_vec.fit(all_addrs)
print('✅ TF-IDF vectorizers fitted')

# Fast lookup dictionaries
s1_name = df1c.set_index('entity_id')['business_name'].fillna('').to_dict()
s1_addr = df1c.set_index('entity_id')['business_address'].fillna('').to_dict()
tg_name = df_target.set_index('entity_id')['business_name'].fillna('').to_dict()
tg_addr = df_target.set_index('entity_id')['business_address'].fillna('').to_dict()

tp = training_pairs.reset_index(drop=True)

# Retrieve strings in batch
n1 = [s1_name.get(i, '') for i in tp['source1_entity_id']]
n2 = [tg_name.get(i, '') for i in tp['target_entity_id']]
a1 = [s1_addr.get(i, '') for i in tp['source1_entity_id']]
a2 = [tg_addr.get(i, '') for i in tp['target_entity_id']]

# Batch TF-IDF cosine similarity (sparse matrix trick)
print('Computing TF-IDF cosine similarities...')
nm1 = name_vec.transform(n1); nm2 = name_vec.transform(n2)
am1 = addr_vec.transform(a1); am2 = addr_vec.transform(a2)
name_tfidf = np.asarray(nm1.multiply(nm2).sum(axis=1)).flatten()
addr_tfidf = np.asarray(am1.multiply(am2).sum(axis=1)).flatten()
print('✅ TF-IDF similarities computed')

def jaccard(s1, s2):
    t1, t2 = set(str(s1).split()), set(str(s2).split())
    if not t1 or not t2: return 0.0
    return len(t1 & t2) / len(t1 | t2)

# Compute all features row by row
print('Computing RapidFuzz + Jaccard features...')
feature_rows = []
for i, row in tqdm(tp.iterrows(), total=len(tp), desc='Feature Gen'):
    feature_rows.append({
        'source1_entity_id'  : row['source1_entity_id'],
        'target_entity_id'   : row['target_entity_id'],
        'label'              : int(row['label']),
        # TF-IDF
        'name_tfidf_sim'     : float(name_tfidf[i]),
        'addr_tfidf_sim'     : float(addr_tfidf[i]),
        # Jaccard
        'name_jaccard'       : jaccard(n1[i], n2[i]),
        'addr_jaccard'       : jaccard(a1[i], a2[i]),
        # RapidFuzz
        'fuzz_ratio'         : fuzz.ratio(n1[i], n2[i]),
        'fuzz_partial_ratio' : fuzz.partial_ratio(n1[i], n2[i]),
        'fuzz_token_sort'    : fuzz.token_sort_ratio(n1[i], n2[i]),
        'fuzz_token_set'     : fuzz.token_set_ratio(n1[i], n2[i]),
    })

features_df = pd.DataFrame(feature_rows)
features_df.to_csv('output/training_dataset.csv', index=False)

print(f'\n✅ FeatureEngineer complete')
print(f'  Shape: {features_df.shape}')
print(f'  Label dist: {features_df.label.value_counts().to_dict()}')
print(features_df.drop(columns=['source1_entity_id','target_entity_id']).describe().round(4).to_string())

---
## Cell 9 — Train LightGBM Classifier

In [ ]:
import lightgbm as lgb
import joblib, json
from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_score, recall_score, fbeta_score, confusion_matrix

FEATURE_COLS = [
    'name_tfidf_sim', 'addr_tfidf_sim',
    'name_jaccard', 'addr_jaccard',
    'fuzz_ratio', 'fuzz_partial_ratio',
    'fuzz_token_sort', 'fuzz_token_set'
]

X = features_df[FEATURE_COLS].astype(float)
y = features_df['label'].astype(int)

if len(np.unique(y)) < 2:
    raise ValueError("Not enough classes to train model. Need both label=0 and label=1.")

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f'Train size: {len(X_train):,}   Val size: {len(X_val):,}')

model = lgb.LGBMClassifier(
    n_estimators=200,
    learning_rate=0.05,
    num_leaves=63,
    max_depth=-1,
    random_state=42,
    n_jobs=-1,
    class_weight='balanced'
)
print('Training LightGBM...')
model.fit(X_train, y_train)

y_pred = model.predict(X_val)
y_prob = model.predict_proba(X_val)[:, 1]

prec = precision_score(y_val, y_pred, zero_division=0)
rec  = recall_score(y_val, y_pred, zero_division=0)
f05  = fbeta_score(y_val, y_pred, beta=0.5, zero_division=0)
cm   = confusion_matrix(y_val, y_pred)

importance = {f: float(i) for f, i in
              sorted(zip(FEATURE_COLS, model.feature_importances_), key=lambda x: x[1], reverse=True)}

print(f'\n📊 Validation Metrics')
print(f'   Precision : {prec:.4f}')
print(f'   Recall    : {rec:.4f}')
print(f'   F0.5 Score: {f05:.4f}')
print(f'\n📉 Confusion Matrix')
print(f'   TN={cm[0][0]}  FP={cm[0][1]}')
print(f'   FN={cm[1][0]}  TP={cm[1][1]}')
print(f'\n🔑 Feature Importance:')
for feat, imp in importance.items():
    bar = '█' * int(imp / max(importance.values()) * 30)
    print(f'   {feat:<25} {bar} {imp:.0f}')

metrics = {
    'precision': float(prec),
    'recall': float(rec),
    'f0.5_score': float(f05),
    'feature_importance': importance
}
with open('output/training_metrics.json', 'w') as f:
    json.dump(metrics, f, indent=4)

joblib.dump(model, 'models/lightgbm_model.pkl')
print('\n✅ Model saved: models/lightgbm_model.pkl')
print('✅ Metrics saved: output/training_metrics.json')

---
## Cell 10 — Optimize Probability Threshold (F0.5 Maximization)

In [ ]:
thresholds = [0.50, 0.55, 0.60, 0.65, 0.70, 0.75, 0.80, 0.85, 0.90, 0.95]
results    = []
best_f05, best_thresh, best_row = -1.0, 0.5, {}

print(f'{'Threshold':>10} | {'Precision':>9} | {'Recall':>8} | {'F0.5':>8}')
print('-' * 48)

for t in thresholds:
    yp = (y_prob >= t).astype(int)
    p  = precision_score(y_val, yp, zero_division=0)
    r  = recall_score(y_val, yp, zero_division=0)
    f  = fbeta_score(y_val, yp, beta=0.5, zero_division=0)
    row = {'threshold': float(t), 'precision': float(p), 'recall': float(r), 'f0.5_score': float(f)}
    results.append(row)
    marker = '  ⭐ BEST' if f > best_f05 else ''
    print(f'{t:>10.2f} | {p:>9.4f} | {r:>8.4f} | {f:>8.4f}{marker}')
    if f > best_f05:
        best_f05, best_thresh, best_row = f, t, row

output_data = {
    'best_threshold': best_thresh,
    'best_metrics': best_row,
    'all_threshold_results': results
}
with open('output/best_threshold.json', 'w') as f:
    json.dump(output_data, f, indent=4)

print(f'\n✅ Best Threshold : {best_thresh}')
print(f'   Best Precision : {best_row["precision"]:.4f}')
print(f'   Best Recall    : {best_row["recall"]:.4f}')
print(f'   Best F0.5      : {best_row["f0.5_score"]:.4f}')
print('✅ Saved: output/best_threshold.json')

---
## Cell 11 — Summary of All Output Files

In [ ]:
print('='*65)
print('  PIPELINE COMPLETE — Output Summary')
print('='*65)

output_files = [
    ('output/candidate_pairs.csv',   'Blocking output — all candidate pairs'),
    ('output/training_dataset.csv',  'Labeled features for training'),
    ('output/training_metrics.json', 'Precision / Recall / F0.5 + feature importance'),
    ('output/best_threshold.json',   'Optimal probability threshold'),
    ('models/lightgbm_model.pkl',    'Trained LightGBM classifier'),
]

for path, desc in output_files:
    if os.path.exists(path):
        size = os.path.getsize(path) / 1024
        print(f'  ✅  {path:<40} ({size:>8.1f} KB)  — {desc}')
    else:
        print(f'  ❌  {path:<40} NOT FOUND')

---
## Cell 12 — Download All Outputs to Your Computer

In [ ]:
from google.colab import files

download_files = [
    'output/candidate_pairs.csv',
    'output/training_dataset.csv',
    'output/training_metrics.json',
    'output/best_threshold.json',
    'models/lightgbm_model.pkl',
]

for f_path in download_files:
    if os.path.exists(f_path):
        files.download(f_path)
        print(f'  ⬇️  Downloaded: {f_path}')
    else:
        print(f'  ⚠️  Skipped (not found): {f_path}')